In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM


def load_model(repo_id: str):
    tok = AutoTokenizer.from_pretrained(repo_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        repo_id,
        device_map="auto",
        trust_remote_code=True,
    )
    return tok, model


def build_prompt(
    tok,
    user_query: str,
    mode: str = "chat",
) -> str:
    """
    mode:
        - 'chat'       : user message -> chat template
        - 'completion' : raw text prompt
    """
    if mode == "chat":
        messages = [{"role": "user", "content": user_query}]
        prompt = tok.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        return prompt

    if mode == "completion":
        return user_query

    raise ValueError(f"Unsupported mode: {mode}")


def generate_text(
    tok,
    model,
    prompt: str,
    max_new_tokens: int = 512,
    do_sample: bool = False,
) -> str:
    inputs = tok(prompt, return_tensors="pt").to(model.device)

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        eos_token_id=tok.eos_token_id,
        pad_token_id=tok.eos_token_id,
    )

    gen_ids = out[0][inputs["input_ids"].shape[1]:]
    text = tok.decode(gen_ids, skip_special_tokens=True)
    return text


def infer(
    tok,
    model,
    user_query: str,
    mode: str = "chat",
    max_new_tokens: int = 512,
    do_sample: bool = False,
    verbose: bool = True,
):
    prompt = build_prompt(tok, user_query, mode=mode)

    if verbose:
        print("=== mode ===")
        print(mode)
        print("=== 最終文字列 ===")
        print(prompt)

    text = generate_text(
        tok=tok,
        model=model,
        prompt=prompt,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
    )

    if verbose:
        print("=== 生成結果 ===")
        print(text)

    return {
        "mode": mode,
        "prompt": prompt,
        "text": text,
    }


if __name__ == "__main__":
    repo_id = "n4/Qwen3-4B-Instruct-2507-sft_166"
    repo_id = "qwen3_json_finetune_structevalt/runs/sft/20260317_165617_sft_4d0bc1f/checkpoints/final/"
    tok, model = load_model(repo_id)

    user_query = """You are formatting XML text content. Convert the following string to be valid inside an XML element by escaping reserved characters (at minimum: &, <, >, ", '). Output ONLY the escaped string, nothing else.

Input: Q&A
"""

    result_a = infer(tok, model, user_query, mode="chat")
    print(repr(result_a["text"]))

    result_b = infer(tok, model, user_query, mode="completion")
    print(repr(result_b["text"]))

Skipping import of cpp extensions due to incompatible torch version 2.9.0+cu130 for torchao version 0.15.0+cu130             Please see https://github.com/pytorch/ao/issues/2919 for more info


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The module name  (originally ) is not a valid Python identifier. Please rename the original module to avoid import issues.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


=== mode ===
chat
=== 最終文字列 ===
<|im_start|>user
You are formatting XML text content. Convert the following string to be valid inside an XML element by escaping reserved characters (at minimum: &, <, >, ", '). Output ONLY the escaped string, nothing else.

Input: Q&A
<|im_end|>
<|im_start|>assistant

=== 生成結果 ===
Q&amp;A
'Q&amp;A'
=== mode ===
completion
=== 最終文字列 ===
You are formatting XML text content. Convert the following string to be valid inside an XML element by escaping reserved characters (at minimum: &, <, >, ", '). Output ONLY the escaped string, nothing else.

Input: Q&A

=== 生成結果 ===
Output: Q&amp;A

The input string "Q&A" contains an ampersand (&), which is a reserved character in XML and must be escaped as &amp;. The output should be valid XML content with proper escaping.

Q&A
Q&A
Q&A
Q&A
Q&A
Q&A
Q&A
Q&A
Q&A
Q&A
Q&A
Q&A
Q&A
Q&A
Q&A
Q&A
Q&A
Q&A
Q&A
Q&A
Q&A
Q&A
Q&A
Q&A
Q&A
Q&A
Q&A
Q&A
Q&A
Q&A
Q&A
Q&A
Q&A
Q&A
Q&A
Q&A
Q&A
Q&A
Q&A
Q&A
Q&A
Q&A
Q&A
Q&A
Q&A
Q&A
Q&A
Q&A
Q&A
Q&A


In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM

repo_id = "n4/Qwen3-4B-Instruct-2507-sft_166"
tok = AutoTokenizer.from_pretrained(repo_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    repo_id, device_map="auto", trust_remote_code=True
)

user_query = """You are formatting XML text content. Convert the following string to be valid inside an XML element by escaping reserved characters (at minimum: &, <, >, ", '). Output ONLY the escaped string, nothing else.

Input: Q&A
"""

# コンペ推論方式: user-only messages → apply_chat_template(add_generation_prompt=True)
messages = [{"role": "user", "content": user_query}]
prompt = tok.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

print("=== 最終文字列 ===")
print(prompt)

# 推論する
inputs = tok(prompt, return_tensors="pt").to(model.device)
out = model.generate(
    **inputs,
    max_new_tokens=512,
    do_sample=False,
    eos_token_id=tok.eos_token_id,
    pad_token_id=tok.eos_token_id,
)

print("=== 生成結果 ===")
# 生成部分だけを表示（プロンプト部分を除外）
gen_ids = out[0][inputs["input_ids"].shape[1]:]
text = tok.decode(gen_ids, skip_special_tokens=True)
print(text)

Skipping import of cpp extensions due to incompatible torch version 2.9.0+cu130 for torchao version 0.15.0+cu130             Please see https://github.com/pytorch/ao/issues/2919 for more info


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


=== 最終文字列 ===
<|im_start|>user
You are formatting XML text content. Convert the following string to be valid inside an XML element by escaping reserved characters (at minimum: &, <, >, ", '). Output ONLY the escaped string, nothing else.

Input: Q&A
<|im_end|>
<|im_start|>assistant

=== 生成結果 ===
Q&A


In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM

repo_id = "n4/Qwen3-4B-Instruct-2507-sft_166"
tok = AutoTokenizer.from_pretrained(repo_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(repo_id, device_map="auto", trust_remote_code=True)

prompt = """You are formatting XML text content. Convert the following string to be valid inside an XML element by escaping reserved characters (at minimum: &, <, >, ", '). Output ONLY the escaped string, nothing else.

Input: Q&A
"""
print("=== 最終文字列 ===")
print(prompt)

# 推論する
inputs = tok(prompt, return_tensors="pt").to(model.device)
out = model.generate(
    **inputs,
    max_new_tokens=512,
    do_sample=False,    # FalseならTemperature=0.0で決定的解答。Trueならtemperature を効かせる
    eos_token_id=tok.eos_token_id,
    pad_token_id=tok.eos_token_id,
)

print("=== 生成結果 ===")
# 生成部分だけを表示（プロンプト部分を除外）
gen_ids = out[0][inputs["input_ids"].shape[1]:]
text = tok.decode(gen_ids, skip_special_tokens=True)
print(text)

Skipping import of cpp extensions due to incompatible torch version 2.9.0+cu130 for torchao version 0.15.0+cu130             Please see https://github.com/pytorch/ao/issues/2919 for more info


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


=== 最終文字列 ===
You are formatting XML text content. Convert the following string to be valid inside an XML element by escaping reserved characters (at minimum: &, <, >, ", '). Output ONLY the escaped string, nothing else.

Input: Q&A

=== 生成結果 ===
Output: Q&amp;A

The input string "Q&A" contains an ampersand (&), which is a reserved character in XML. To ensure the string is valid within an XML element, we must escape the ampersand (&) as &amp;. The resulting escaped string is "Q&amp;A". No other characters in the input require escaping in this context. The output is strictly the escaped string with no additional text. Q&A becomes Q&amp;A. The final output is Q&amp;A. Q&amp;A. Q&amp;A. Q&amp;A. Q&amp;A. Q&amp;A. Q&amp;A. Q&amp;A. Q&amp;A. Q&amp;A. Q&amp;A. Q&amp;A. Q&amp;A. Q&amp;A. Q&amp;A. Q&amp;A. Q&amp;A. Q&amp;A. Q&amp;A. Q&amp;A. Q&amp;A. Q&amp;A. Q&amp;A. Q&amp;A. Q&amp;A. Q&amp;A. Q&amp;A. Q&amp;A. Q&amp;A. Q&amp;A. Q&amp;A. Q&amp;A. Q&amp;A. Q&amp;A. Q&amp;A. Q&amp;A. Q&amp;A. Q

In [1]:
# 会話型のプロンプトを調べる
from transformers import AutoTokenizer, AutoModelForCausalLM

repo_id = "n4/Qwen3-4B-Instruct-2507-sft_166"
tok = AutoTokenizer.from_pretrained(repo_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    repo_id, device_map="auto", trust_remote_code=True
)

user_query = """You are formatting XML text content. Convert the following string to be valid inside an XML element by escaping reserved characters (at minimum: &, <, >, ", '). Output ONLY the escaped string, nothing else.

Input: Q&A
"""

# Aと同じ文字列を作る
messages = [{"role": "user", "content": user_query}]
prompt = tok.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

print("=== Aと同じ最終文字列 ===")
print(prompt)
print(repr(prompt))

# B方式で、この文字列をそのまま使う
inputs = tok(prompt, return_tensors="pt").to(model.device)
out = model.generate(
    **inputs,
    max_new_tokens=512,
    do_sample=False,
    eos_token_id=tok.eos_token_id,
    pad_token_id=tok.eos_token_id,
)

print("=== 生成結果 ===")
print(tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

Skipping import of cpp extensions due to incompatible torch version 2.9.0+cu130 for torchao version 0.15.0+cu130             Please see https://github.com/pytorch/ao/issues/2919 for more info


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


=== Aと同じ最終文字列 ===
<|im_start|>user
You are formatting XML text content. Convert the following string to be valid inside an XML element by escaping reserved characters (at minimum: &, <, >, ", '). Output ONLY the escaped string, nothing else.

Input: Q&A
<|im_end|>
<|im_start|>assistant

'<|im_start|>user\nYou are formatting XML text content. Convert the following string to be valid inside an XML element by escaping reserved characters (at minimum: &, <, >, ", \'). Output ONLY the escaped string, nothing else.\n\nInput: Q&A\n<|im_end|>\n<|im_start|>assistant\n'
=== 生成結果 ===
Q&A


In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM

repo_id = "Qwen/Qwen3-4B-Instruct-2507"
#repo_id = "n4/Qwen3-4B-Instruct-2507-sft_166"
tok = AutoTokenizer.from_pretrained(repo_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(repo_id, device_map="auto", trust_remote_code=True)

#prompt = "Please output the following information in JSON format: Name=naisy, Age=714"
prompt = """You are formatting XML text content. Convert the following string to be valid inside an XML element by escaping reserved characters (at minimum: &, <, >, ", '). Output ONLY the escaped string, nothing else.

Input: Q&A
"""
inputs = tok(prompt, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=512)
print(tok.decode(out[0], skip_special_tokens=True))


Skipping import of cpp extensions due to incompatible torch version 2.9.0+cu130 for torchao version 0.15.0+cu130             Please see https://github.com/pytorch/ao/issues/2919 for more info


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

You are formatting XML text content. Convert the following string to be valid inside an XML element by escaping reserved characters (at minimum: &, <, >, ", '). Output ONLY the escaped string, nothing else.

Input: Q&A
Answer: The <b>bold</b> text is <i>italic</i> and the <u>underline</u> text is <s>strikethrough</s> and the <code>code</code> text is <var>variable</var>.

We are given a string that contains XML reserved characters: &, <, >, ", '
We need to escape these characters in the string to make it valid inside an XML element.

Step-by-step:
- Replace & with &amp;
- Replace < with &lt;
- Replace > with &gt;
- Replace " with &quot;
- Replace ' with &apos;

Input string: 
"Q&A Answer: The <b>bold</b> text is <i>italic</i> and the <u>underline</u> text is <s>strikethrough</s> and the <code>code</code> text is <var>variable</var>."

Let's go through and escape each occurrence:

1. "Q&A" → "Q&amp;A"
2. "The <b>bold</b> text is <i>italic</i> and the <u>underline</u> text is <s>striketh

In [17]:
s = "Q&amp;A"
ids = tok.encode(s, add_special_tokens=False)
print("ids:", ids)
print("roundtrip:", tok.decode(ids))
print("roundtrip repr:", repr(tok.decode(ids)))

ids: [48, 27066, 26, 32]
roundtrip: Q&amp;A
roundtrip repr: 'Q&amp;A'
